# Direct vLLM evaluation of three Fed-MOPSD Client Students

This notebook follows `client_math_eval_vllm.ipynb`: vLLM is created directly in the Papermill kernel, without generated Python workers. Clients run sequentially as Math → IF → Code. AIME25 and AIME26 prompts are interleaved in the same vLLM request batches and reported separately as Avg@12.

In [ ]:
# Papermill parameters
GPU_ID = "0"
CLIENT_MODELS_ROOT = "results/fed_mopsd_qwen3_4b_base/training/round_000"
DATA_ROOT = "data"
OUTPUT_DIR = "results/fed_mopsd_qwen3_4b_base/client_student_evaluation_vllm_38912"
LCB_REPO_DIR = "data/code/LiveCodeBench"
LCB_LOCAL_DATA_DIR = "data/code/eval"
NLTK_DATA_DIR = "data/nltk_data"

LOCAL_FILES_ONLY = True
RESUME = True
SEED = 42

MAX_MODEL_LEN = 40960
MAX_NEW_TOKENS = 38912
ENABLE_THINKING = True
TEMPERATURE = 1.0
TOP_P = 1.0
TOP_K = -1
MIN_P = 0.0
PRESENCE_PENALTY = 0.0

MATH_SAMPLES_PER_PROMPT = 12
IF_SAMPLES_PER_PROMPT = 1
CODE_SAMPLES_PER_PROMPT = 1
MATH_REQUEST_BATCH_SIZE = 4
IF_REQUEST_BATCH_SIZE = 8
CODE_REQUEST_BATCH_SIZE = 8

VLLM_GPU_MEMORY_UTILIZATION = 0.85
VLLM_MAX_NUM_SEQS = 12
VLLM_ENFORCE_EAGER = True

LIMIT_AIME25 = None
LIMIT_AIME26 = None
LIMIT_IFBENCH = None
LIMIT_IFEVAL = None
LIMIT_LCB = None

LCB_RELEASE_VERSION = "release_v6"
LCB_DATASET_REVISION = "d44be6b144381afa49392b1f0eb424a64a4d8a10"
LCB_REPO_REVISION = "28fef95ea8c9f7a547c8329f2cd3d32b92c1fa24"
LCB_NUM_PROCESSES = 4
LCB_TIMEOUT_SECONDS = 6


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ["VLLM_ALLOW_LONG_MAX_MODEL_LEN"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

import gc
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import re
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from datasets import disable_progress_bars, load_dataset
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

disable_progress_bars()
workspace = Path.cwd().resolve()
client_root = Path(CLIENT_MODELS_ROOT).expanduser()
if not client_root.is_absolute():
    client_root = (workspace / client_root).resolve()
output_dir = Path(OUTPUT_DIR).expanduser()
if not output_dir.is_absolute():
    output_dir = (workspace / output_dir).resolve()
output_dir.mkdir(parents=True, exist_ok=True)
data_root = Path(DATA_ROOT).expanduser()
if not data_root.is_absolute():
    data_root = (workspace / data_root).resolve()

LOCAL_EVAL_FILES = {
    "aime25": data_root / "math" / "eval" / "aime25.parquet",
    "aime26": data_root / "math" / "eval" / "aime26.parquet",
    "ifbench": data_root / "if" / "eval" / "ifbench.parquet",
    "ifeval": data_root / "if" / "eval" / "ifeval.parquet",
}
missing_eval_files = [str(path) for path in LOCAL_EVAL_FILES.values() if not path.is_file()]
if missing_eval_files:
    raise FileNotFoundError("Missing local evaluation data: " + ", ".join(missing_eval_files))

MODEL_DIRS = {
    domain: (client_root / f"client_{domain}" / "student_model").resolve()
    for domain in ("math", "if", "code")
}
RAW_OUTPUT_FILES = {
    "aime25": output_dir / "math" / "raw" / "aime25.jsonl",
    "aime26": output_dir / "math" / "raw" / "aime26.jsonl",
    "ifbench": output_dir / "if" / "raw" / "ifbench.jsonl",
    "ifeval": output_dir / "if" / "raw" / "ifeval.jsonl",
    "livecodebench": output_dir / "code" / "raw" / "livecodebench.jsonl",
}
for raw_path in RAW_OUTPUT_FILES.values():
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    if not RESUME:
        raw_path.unlink(missing_ok=True)
    raw_path.touch(exist_ok=True)
print("Client models:", json.dumps({k: str(v) for k, v in MODEL_DIRS.items()}, indent=2), flush=True)
print(f"Data root: {data_root}", flush=True)
print("Local evaluation data:", json.dumps({k: str(v) for k, v in LOCAL_EVAL_FILES.items()}, indent=2), flush=True)
print("Raw outputs:", json.dumps({k: str(v) for k, v in RAW_OUTPUT_FILES.items()}, indent=2), flush=True)
print(f"Output: {output_dir}", flush=True)


def format_seconds(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def atomic_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def append_jsonl(path, record):
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def record_key(sample_id, sample_index):
    return f"{sample_id}::sample-{int(sample_index):02d}"


def model_signature(model_dir):
    digest = hashlib.sha256()
    for path in sorted(Path(model_dir).glob("*.safetensors")):
        stat = path.stat()
        digest.update(f"{path.name}:{stat.st_size}:{stat.st_mtime_ns}".encode("utf-8"))
    for name in ("config.json", "model.safetensors.index.json"):
        path = Path(model_dir) / name
        if path.is_file():
            digest.update(path.read_bytes())
    return digest.hexdigest()[:16]


def make_fingerprint(domain, model_dir, samples_per_prompt):
    payload = {
        "method": "direct-vllm-client-eval-v1",
        "domain": domain,
        "model_signature": model_signature(model_dir),
        "max_model_len": int(MAX_MODEL_LEN),
        "max_new_tokens": int(MAX_NEW_TOKENS),
        "enable_thinking": bool(ENABLE_THINKING),
        "temperature": float(TEMPERATURE),
        "top_p": float(TOP_P),
        "top_k": int(TOP_K),
        "min_p": float(MIN_P),
        "samples_per_prompt": int(samples_per_prompt),
        "seed": int(SEED),
        "vllm_version": importlib.metadata.version("vllm"),
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:16], payload


def load_records(path, run_fingerprint):
    records = {}
    path = Path(path)
    if not path.is_file():
        return records
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if record.get("run_fingerprint") == run_fingerprint:
                records[record_key(record["sample_id"], record["sample_index"])] = record
    return records


def split_thinking(text):
    text = str(text).strip()
    if "</think>" in text:
        thinking, final_answer = text.rsplit("</think>", 1)
        thinking = re.sub(r"^\s*<think>\s*", "", thinking).strip()
        return thinking, final_answer.strip(), True
    return "", text, False


def stable_seed(domain, benchmark, sample_id):
    digest = hashlib.sha256(f"{domain}:{benchmark}:{sample_id}".encode("utf-8")).digest()
    return (int(SEED) + int.from_bytes(digest[:4], "big")) % (2**31 - 1)


def load_vllm(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir, local_files_only=bool(LOCAL_FILES_ONLY), trust_remote_code=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    llm = LLM(
        model=str(model_dir),
        tokenizer=str(model_dir),
        dtype="bfloat16",
        tensor_parallel_size=1,
        max_model_len=int(MAX_MODEL_LEN),
        gpu_memory_utilization=float(VLLM_GPU_MEMORY_UTILIZATION),
        max_num_seqs=int(VLLM_MAX_NUM_SEQS),
        trust_remote_code=True,
        enforce_eager=bool(VLLM_ENFORCE_EAGER),
        enable_prefix_caching=True,
        seed=int(SEED),
    )
    return llm, tokenizer


def release_vllm(llm, tokenizer, label):
    engine = getattr(llm, "llm_engine", None)
    shutdown = getattr(engine, "shutdown", None)
    if callable(shutdown):
        shutdown()
    del llm, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(2)
    print(f"[UNLOAD] {label} vLLM engine released", flush=True)


def render_prompt(tokenizer, messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=bool(ENABLE_THINKING),
    )


def run_entries(llm, tokenizer, domain, entries, samples_per_prompt, batch_size, run_fingerprint, scorer):
    benchmarks = list(dict.fromkeys(entry["benchmark"] for entry in entries))
    paths = {name: RAW_OUTPUT_FILES[name] for name in benchmarks}
    records = {
        name: load_records(paths[name], run_fingerprint) if RESUME else {}
        for name in benchmarks
    }
    completed = {
        name: {key for key, record in values.items() if record.get("status") == "ok"}
        for name, values in records.items()
    }
    totals = {
        name: sum(entry["benchmark"] == name for entry in entries) * int(samples_per_prompt)
        for name in benchmarks
    }
    done = {
        name: sum(1 for key in completed[name])
        for name in benchmarks
    }
    pending = []
    for entry in entries:
        keys = {record_key(entry["sample_id"], j) for j in range(samples_per_prompt)}
        if not keys.issubset(completed[entry["benchmark"]]):
            pending.append(entry)
    for name in benchmarks:
        if done[name]:
            print(f"[RESUME] {domain}/{name}: {done[name]}/{totals[name]}", flush=True)
    if not pending:
        for name in benchmarks:
            print(f"[SKIP] {domain}/{name}: {done[name]}/{totals[name]} complete", flush=True)
        return records

    started = time.time()
    for chunk_start in range(0, len(pending), int(batch_size)):
        chunk = pending[chunk_start:chunk_start + int(batch_size)]
        prompts, params, prepared = [], [], []
        for entry in chunk:
            rendered = render_prompt(tokenizer, entry["messages"])
            prompt_ids = tokenizer(rendered, add_special_tokens=False)["input_ids"]
            effective_max_tokens = min(int(MAX_NEW_TOKENS), int(MAX_MODEL_LEN) - len(prompt_ids))
            request_seed = stable_seed(domain, entry["benchmark"], entry["sample_id"])
            prompts.append(rendered)
            params.append(SamplingParams(
                n=int(samples_per_prompt),
                temperature=float(TEMPERATURE),
                top_p=float(TOP_P),
                top_k=int(TOP_K),
                min_p=float(MIN_P),
                presence_penalty=float(PRESENCE_PENALTY),
                max_tokens=int(effective_max_tokens),
                seed=int(request_seed),
                skip_special_tokens=True,
            ))
            prepared.append({
                **entry,
                "input_tokens": len(prompt_ids),
                "effective_max_new_tokens": effective_max_tokens,
                "request_seed": request_seed,
            })

        chunk_started = time.time()
        outputs = llm.generate(prompts, sampling_params=params, use_tqdm=True)
        chunk_seconds = time.time() - chunk_started
        for entry, request_output in zip(prepared, outputs):
            benchmark = entry["benchmark"]
            for choice in sorted(request_output.outputs, key=lambda item: int(item.index)):
                sample_index = int(choice.index)
                key = record_key(entry["sample_id"], sample_index)
                if key in completed[benchmark]:
                    continue
                raw_generation = str(choice.text)
                thinking, final_answer, found_think_close = split_thinking(raw_generation)
                record = {
                    "run_fingerprint": run_fingerprint,
                    "backend": "vllm",
                    "domain": domain,
                    "benchmark": benchmark,
                    "sample_id": entry["sample_id"],
                    "sample_index": sample_index,
                    "samples_per_prompt": int(samples_per_prompt),
                    "request_seed": entry["request_seed"],
                    "raw_generation": raw_generation,
                    "thinking": thinking,
                    "final_answer": final_answer,
                    "found_think_close": found_think_close,
                    "input_tokens": entry["input_tokens"],
                    "output_tokens": len(choice.token_ids),
                    "effective_max_new_tokens": entry["effective_max_new_tokens"],
                    "finish_reason": choice.finish_reason,
                    "chunk_generation_seconds": chunk_seconds,
                    "status": "ok",
                    **entry["metadata"],
                }
                record.update(scorer(entry, record))
                append_jsonl(paths[benchmark], record)
                records[benchmark][key] = record
                completed[benchmark].add(key)
                done[benchmark] += 1
            print(
                f"[PROGRESS] {domain}/{benchmark}: {done[benchmark]}/{totals[benchmark]} "
                f"({100.0 * done[benchmark] / totals[benchmark]:.2f}%) "
                f"elapsed={format_seconds(time.time() - started)}",
                flush=True,
            )
    return {
        name: load_records(paths[name], run_fingerprint)
        for name in benchmarks
    }


def metric_row(benchmark, domain, metric, numerator, denominator, note=""):
    score = numerator / denominator if denominator else None
    return {
        "benchmark": benchmark,
        "domain": domain,
        "metric": metric,
        "score": score,
        "score_percent": 100.0 * score if score is not None else None,
        "numerator": numerator,
        "denominator": denominator,
        "status": "complete",
        "note": note,
    }


def save_domain_metrics(domain, model_dir, run_fingerprint, protocol, rows, extra=None):
    domain_dir = output_dir / domain
    domain_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "domain": domain,
        "model_dir": str(model_dir),
        "run_fingerprint": run_fingerprint,
        "protocol": protocol,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": rows,
    }
    if extra:
        payload.update(extra)
    atomic_json(domain_dir / "metrics.json", payload)
    pd.DataFrame(rows).to_csv(domain_dir / "metrics.csv", index=False, encoding="utf-8-sig")


In [ ]:
# Math Client: AIME25 and AIME26 are interleaved in the same vLLM batches.
from math_verify import parse as math_parse, verify as math_verify


def limited(dataset, limit):
    if limit is None:
        return dataset
    return dataset.select(range(min(int(limit), len(dataset))))


def extract_last_boxed(text):
    starts = [match.start() for match in re.finditer(r"\\boxed\s*\{", str(text))]
    for start in reversed(starts):
        brace_start = str(text).find("{", start)
        depth = 0
        for index in range(brace_start, len(str(text))):
            if str(text)[index] == "{":
                depth += 1
            elif str(text)[index] == "}":
                depth -= 1
                if depth == 0:
                    return str(text)[brace_start + 1:index].strip()
    return None


def normalize_math(value):
    value = str(value).strip().replace("$", "").replace(",", "")
    value = re.sub(r"\\(?:mathrm|text)\{([^{}]*)\}", r"\1", value)
    value = re.sub(r"\s+", "", value)
    if re.fullmatch(r"[+-]?\d+\.0+", value):
        value = str(int(float(value)))
    if re.fullmatch(r"[+-]?\d+", value):
        value = str(int(value))
    return value


def grade_math(predicted, gold):
    if predicted is None:
        return False
    try:
        predicted_text = predicted if "$" in predicted else f"${predicted}$"
        gold_text = str(gold) if "$" in str(gold) else f"${gold}$"
        return bool(math_verify(
            math_parse(gold_text, fallback_mode="no_fallback"),
            math_parse(predicted_text, fallback_mode="no_fallback"),
            timeout_seconds=5,
        ))
    except Exception:
        return normalize_math(predicted) == normalize_math(gold)


def score_math(entry, record):
    predicted = extract_last_boxed(record["raw_generation"])
    return {
        "predicted_answer": predicted or "",
        "formatted": predicted is not None,
        "correct": grade_math(predicted, entry["metadata"]["gold_answer"]),
    }


math_sets = {
    "aime25": limited(load_dataset(
        "parquet", data_files={"eval": str(LOCAL_EVAL_FILES["aime25"])}, split="eval"
    ), LIMIT_AIME25),
    "aime26": limited(load_dataset(
        "parquet", data_files={"eval": str(LOCAL_EVAL_FILES["aime26"])}, split="eval"
    ), LIMIT_AIME26),
}
math_by_benchmark = {}
for benchmark, dataset in math_sets.items():
    entries = []
    for index, row in enumerate(dataset):
        sample_id = str(row.get("id", index))
        prompt = str(row["problem"]).strip() + "\n\nPlease reason step by step, and put your final answer within \\boxed{}."
        entries.append({
            "benchmark": benchmark,
            "sample_id": sample_id,
            "messages": [{"role": "user", "content": prompt}],
            "metadata": {"problem": str(row["problem"]), "gold_answer": str(row["answer"])},
        })
    math_by_benchmark[benchmark] = entries

# Alternate AIME25 and AIME26 so each request batch contains both benchmarks.
math_entries = []
for index in range(max(len(values) for values in math_by_benchmark.values())):
    for benchmark in ("aime25", "aime26"):
        if index < len(math_by_benchmark[benchmark]):
            math_entries.append(math_by_benchmark[benchmark][index])

math_model_dir = MODEL_DIRS["math"]
math_fp, math_protocol = make_fingerprint("math", math_model_dir, MATH_SAMPLES_PER_PROMPT)
print(f"[LOAD] Math Client: {math_model_dir}; fingerprint={math_fp}", flush=True)
math_llm, math_tokenizer = load_vllm(math_model_dir)
math_records = run_entries(
    math_llm, math_tokenizer, "math", math_entries,
    MATH_SAMPLES_PER_PROMPT, MATH_REQUEST_BATCH_SIZE,
    math_fp, score_math,
)

math_metrics = []
for benchmark, entries in math_by_benchmark.items():
    selected = [
        math_records[benchmark][record_key(entry["sample_id"], sample_index)]
        for entry in entries
        for sample_index in range(MATH_SAMPLES_PER_PROMPT)
    ]
    correct = sum(bool(record.get("correct")) for record in selected)
    formatted = sum(bool(record.get("formatted")) for record in selected)
    passed = sum(
        any(bool(math_records[benchmark][record_key(entry["sample_id"], j)].get("correct")) for j in range(MATH_SAMPLES_PER_PROMPT))
        for entry in entries
    )
    math_metrics.extend([
        metric_row(benchmark.upper(), "math", "avg_at_12", correct, len(selected), f"format_rate={formatted}/{len(selected)}"),
        metric_row(benchmark.upper(), "math", "pass_at_12", passed, len(entries)),
    ])
save_domain_metrics("math", math_model_dir, math_fp, math_protocol, math_metrics)
display(pd.DataFrame(math_metrics))
release_vllm(math_llm, math_tokenizer, "Math Client")
del math_llm, math_tokenizer


In [ ]:
# Instruction-Following Client: IFBench and IFEval.
nltk_data_dir = Path(NLTK_DATA_DIR).expanduser().resolve()
os.environ["NLTK_DATA"] = str(nltk_data_dir)
import nltk
if os.environ["NLTK_DATA"] not in nltk.data.path:
    nltk.data.path.insert(0, os.environ["NLTK_DATA"])
required_nltk_resources = {
    "punkt_tab": "tokenizers/punkt_tab",
    "stopwords": "corpora/stopwords",
    "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng",
}
for package_name, resource_path in required_nltk_resources.items():
    try:
        located = nltk.data.find(resource_path)
        print(f"[NLTK] {package_name}: {located}", flush=True)
    except LookupError as exc:
        raise FileNotFoundError(
            f"Missing NLTK resource {package_name!r} under {nltk_data_dir}. "
            f"Run this notebook with NLTK_DATA_DIR={nltk_data_dir}."
        ) from exc
from ifbench import instructions_registry


def response_variants(response):
    lines = response.split("\n")
    remove_first = "\n".join(lines[1:]).strip()
    remove_last = "\n".join(lines[:-1]).strip()
    remove_both = "\n".join(lines[1:-1]).strip()
    return [
        response, response.replace("*", ""), remove_first, remove_last, remove_both,
        remove_first.replace("*", ""), remove_last.replace("*", ""), remove_both.replace("*", ""),
    ]


def check_instruction(instruction_id, kwargs, prompt, candidates):
    instruction_cls = instructions_registry.INSTRUCTION_DICT[instruction_id]
    instruction = instruction_cls(instruction_id)
    instruction.build_description(**{key: value for key, value in dict(kwargs).items() if value is not None})
    args = instruction.get_instruction_args()
    if args and "prompt" in args:
        instruction.build_description(prompt=prompt)
    return any(candidate.strip() and instruction.check_following(candidate) for candidate in candidates)


def score_if(entry, record):
    response = record["final_answer"]
    metadata = entry["metadata"]
    strict, loose = [], []
    for instruction_id, kwargs in zip(metadata["instruction_id_list"], metadata["kwargs"]):
        strict.append(check_instruction(instruction_id, kwargs, metadata["prompt"], [response]))
        loose.append(check_instruction(instruction_id, kwargs, metadata["prompt"], response_variants(response)))
    return {
        "strict_instruction_results": strict,
        "loose_instruction_results": loose,
        "strict_prompt_pass": all(strict),
        "loose_prompt_pass": all(loose),
    }


if_sets = {
    "ifbench": limited(load_dataset(
        "parquet", data_files={"eval": str(LOCAL_EVAL_FILES["ifbench"])}, split="eval"
    ), LIMIT_IFBENCH),
    "ifeval": limited(load_dataset(
        "parquet", data_files={"eval": str(LOCAL_EVAL_FILES["ifeval"])}, split="eval"
    ), LIMIT_IFEVAL),
}
if_entries_by_benchmark = {}
if_entries = []
for benchmark, dataset in if_sets.items():
    entries = []
    for index, row in enumerate(dataset):
        sample_id = str(row.get("key", index))
        metadata = {
            "prompt": str(row["prompt"]),
            "instruction_id_list": list(row["instruction_id_list"]),
            "kwargs": [dict(item) for item in row["kwargs"]],
        }
        entries.append({
            "benchmark": benchmark,
            "sample_id": sample_id,
            "messages": [{"role": "user", "content": metadata["prompt"]}],
            "metadata": metadata,
        })
    if_entries_by_benchmark[benchmark] = entries
    if_entries.extend(entries)

if_model_dir = MODEL_DIRS["if"]
if_fp, if_protocol = make_fingerprint("if", if_model_dir, IF_SAMPLES_PER_PROMPT)
print(f"[LOAD] IF Client: {if_model_dir}; fingerprint={if_fp}", flush=True)
if_llm, if_tokenizer = load_vllm(if_model_dir)
if_records = run_entries(
    if_llm, if_tokenizer, "if", if_entries,
    IF_SAMPLES_PER_PROMPT, IF_REQUEST_BATCH_SIZE,
    if_fp, score_if,
)

if_metrics = []
for benchmark, entries in if_entries_by_benchmark.items():
    selected = [if_records[benchmark][record_key(entry["sample_id"], 0)] for entry in entries]
    strict_lists = [record["strict_instruction_results"] for record in selected]
    loose_lists = [record["loose_instruction_results"] for record in selected]
    instruction_total = sum(len(values) for values in strict_lists)
    if_metrics.extend([
        metric_row(benchmark.upper(), "if", "prompt_level_strict", sum(bool(v) and all(v) for v in strict_lists), len(selected)),
        metric_row(benchmark.upper(), "if", "instruction_level_strict", sum(sum(v) for v in strict_lists), instruction_total),
        metric_row(benchmark.upper(), "if", "prompt_level_loose", sum(bool(v) and all(v) for v in loose_lists), len(selected)),
        metric_row(benchmark.upper(), "if", "instruction_level_loose", sum(sum(v) for v in loose_lists), instruction_total),
    ])
save_domain_metrics("if", if_model_dir, if_fp, if_protocol, if_metrics)
display(pd.DataFrame(if_metrics))
release_vllm(if_llm, if_tokenizer, "IF Client")
del if_llm, if_tokenizer


In [ ]:
# Code Client: LiveCodeBench release_v6 generation.
lcb_repo = Path(LCB_REPO_DIR).expanduser()
if not lcb_repo.is_absolute():
    lcb_repo = (workspace / lcb_repo).resolve()
if not (lcb_repo / "lcb_runner").is_dir():
    if lcb_repo.exists() and any(lcb_repo.iterdir()):
        raise RuntimeError(f"LiveCodeBench directory exists but is incomplete: {lcb_repo}")
    lcb_repo.parent.mkdir(parents=True, exist_ok=True)
    print(f"[CLONE] LiveCodeBench -> {lcb_repo}", flush=True)
    subprocess.run(
        ["git", "clone", "https://github.com/LiveCodeBench/LiveCodeBench.git", str(lcb_repo)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(lcb_repo), "checkout", "--detach", LCB_REPO_REVISION],
        check=True,
    )
print(f"LiveCodeBench scorer: {lcb_repo}", flush=True)
if str(lcb_repo) not in sys.path:
    sys.path.insert(0, str(lcb_repo))

from lcb_runner.benchmarks.code_generation import CodeGenerationProblem
from lcb_runner.lm_styles import LMStyle
from lcb_runner.utils.extraction_utils import extract_code

local_lcb = Path(LCB_LOCAL_DATA_DIR).expanduser()
if not local_lcb.is_absolute():
    local_lcb = (workspace / local_lcb).resolve()
lcb_files = [local_lcb / f"test-{index:05d}-of-00009.parquet" for index in range(9)]
missing_lcb_files = [str(path) for path in lcb_files if not path.is_file()]
if missing_lcb_files:
    raise FileNotFoundError("Missing local LiveCodeBench shards: " + ", ".join(missing_lcb_files))
print(f"LiveCodeBench data: {len(lcb_files)}/9 local shards from {local_lcb}", flush=True)

raw_lcb = load_dataset(
    "parquet", data_files={"test": [str(path) for path in lcb_files]},
    split="test",
)
lcb_problems = sorted([CodeGenerationProblem(**row) for row in raw_lcb], key=lambda item: str(item.question_id))
if LIMIT_LCB is not None:
    lcb_problems = lcb_problems[:min(int(LIMIT_LCB), len(lcb_problems))]

prompt_path = lcb_repo / "lcb_runner/prompts/code_generation.py"
prompt_spec = importlib.util.spec_from_file_location("_lcb_prompt", prompt_path)
prompt_module = importlib.util.module_from_spec(prompt_spec)
notebook_cwd = Path.cwd()
try:
    os.chdir(lcb_repo)
    prompt_spec.loader.exec_module(prompt_module)
finally:
    os.chdir(notebook_cwd)
PromptConstants = prompt_module.PromptConstants
get_generic_question_template_answer = prompt_module.get_generic_question_template_answer


def score_code(entry, record):
    return {"extracted_code": extract_code(record["final_answer"], LMStyle.CodeQwenInstruct)}


code_entries = []
for problem in lcb_problems:
    code_entries.append({
        "benchmark": "livecodebench",
        "sample_id": str(problem.question_id),
        "messages": [
            {"role": "system", "content": PromptConstants.SYSTEM_MESSAGE_GENERIC},
            {"role": "user", "content": get_generic_question_template_answer(problem)},
        ],
        "metadata": {
            "question_title": problem.question_title,
            "question_content": problem.question_content,
            "platform": problem.platform.value,
            "contest_id": problem.contest_id,
            "contest_date": problem.contest_date.isoformat(),
            "difficulty": problem.difficulty.value,
            "starter_code": problem.starter_code,
        },
    })

code_model_dir = MODEL_DIRS["code"]
code_fp, code_protocol = make_fingerprint("code", code_model_dir, CODE_SAMPLES_PER_PROMPT)
print(f"[LOAD] Code Client: {code_model_dir}; fingerprint={code_fp}", flush=True)
code_llm, code_tokenizer = load_vllm(code_model_dir)
code_records_by_benchmark = run_entries(
    code_llm, code_tokenizer, "code", code_entries,
    CODE_SAMPLES_PER_PROMPT, CODE_REQUEST_BATCH_SIZE,
    code_fp, score_code,
)
code_records = code_records_by_benchmark["livecodebench"]
release_vllm(code_llm, code_tokenizer, "Code Client")
del code_llm, code_tokenizer


In [ ]:
# Official LiveCodeBench pass@1 scoring after the Code vLLM engine is released.
class LineProgress:
    def __init__(self, *args, total=None, **kwargs):
        self.total = int(total or 0)
        self.done = 0
        self.started = time.time()
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc, traceback):
        return False
    def update(self, amount=1):
        self.done += int(amount)
        print(
            f"[PROGRESS] code/livecodebench-check: {self.done}/{self.total} "
            f"elapsed={format_seconds(time.time() - self.started)}",
            flush=True,
        )


import lcb_runner.evaluation.compute_code_generation_metrics as lcb_metric_module
from lcb_runner.evaluation.pass_k_utils import extract_instance_results
lcb_metric_module.tqdm = LineProgress

samples_list = [problem.get_evaluation_sample() for problem in lcb_problems]
generations_list = [[code_records[record_key(str(problem.question_id), 0)]["extracted_code"]] for problem in lcb_problems]
official_metrics, official_results, official_metadata = lcb_metric_module.codegen_metrics(
    samples_list,
    generations_list,
    k_list=[1],
    num_process_evaluate=int(LCB_NUM_PROCESSES),
    timeout=int(LCB_TIMEOUT_SECONDS),
    debug=False,
)
instance_grades = extract_instance_results(official_results)
graded = [bool(values[0]) for values in instance_grades]
correct = sum(graded)
total = len(graded)
pass_at_1 = float(official_metrics["pass@1"])

code_domain_dir = output_dir / "code"
atomic_json(code_domain_dir / "livecodebench_eval.json", {
    "run_fingerprint": code_fp,
    "official_metrics": official_metrics,
    "correct": correct,
    "total": total,
    "pass@1": pass_at_1,
    "per_problem": [
        {
            "question_id": str(lcb_problems[index].question_id),
            "passed": graded[index],
            "test_results": official_results[index][0],
            "metadata": official_metadata[index][0],
        }
        for index in range(total)
    ],
})
code_metrics = [metric_row("LiveCodeBench", "code", "pass@1", correct, total)]
save_domain_metrics(
    "code", code_model_dir, code_fp, code_protocol, code_metrics,
    {"official_metrics": official_metrics},
)
display(pd.DataFrame(code_metrics))


In [ ]:
all_metrics = []
domain_payloads = {}
for domain in ("math", "if", "code"):
    payload = json.loads((output_dir / domain / "metrics.json").read_text(encoding="utf-8"))
    domain_payloads[domain] = payload
    all_metrics.extend(payload["metrics"])

metrics = pd.DataFrame(all_metrics)
metrics.to_csv(output_dir / "client_students_vllm_metrics.csv", index=False, encoding="utf-8-sig")
atomic_json(output_dir / "client_students_vllm_metrics.json", {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "domains": domain_payloads,
    "metrics": all_metrics,
})
display(metrics)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

specs = [
    ("AIME25", "avg_at_12", "AIME25\nAvg@12"),
    ("AIME26", "avg_at_12", "AIME26\nAvg@12"),
    ("IFBENCH", "prompt_level_loose", "IFBench\nPrompt loose"),
    ("IFEVAL", "prompt_level_loose", "IFEval\nPrompt loose"),
    ("LiveCodeBench", "pass@1", "LiveCodeBench\nPass@1"),
]
labels, values = [], []
for benchmark, metric, label in specs:
    row = metrics[(metrics["benchmark"] == benchmark) & (metrics["metric"] == metric)]
    if not row.empty:
        labels.append(label)
        values.append(float(row.iloc[0]["score_percent"]))

figure, axis = plt.subplots(figsize=(14, 7.5))
bars = axis.bar(
    range(len(labels)), values,
    color=["#4C78A8", "#4C78A8", "#F58518", "#F58518", "#54A24B"][:len(labels)],
    edgecolor="#333333",
)
for bar, value in zip(bars, values):
    axis.text(
        bar.get_x() + bar.get_width() / 2, value + 1.2, f"{value:.2f}%",
        ha="center", va="bottom", fontsize=12,
    )
axis.set_xticks(range(len(labels)), labels)
axis.tick_params(axis="x", labelsize=12)
axis.tick_params(axis="y", labelsize=11)
axis.set_ylabel("Score (%)", fontsize=12)
axis.set_ylim(0, 108)
axis.set_title("Fed-MOPSD Qwen3-4B: Client Students Evaluation", fontsize=16)
axis.grid(axis="y", linestyle="--", alpha=0.3)
figure.tight_layout()
chart_path = output_dir / "fed_mopsd_client_students_scores.png"
figure.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Metrics: {output_dir / 'client_students_vllm_metrics.json'}", flush=True)
print(f"Chart:   {chart_path}", flush=True)
print("[DONE] Math, IF, and Code Client evaluations complete.", flush=True)


## Papermill

```bash
CUDA_VISIBLE_DEVICES=0 papermill eval_client_students_vllm.ipynb eval_client_students_vllm_running.ipynb --cwd /workspace/code --request-save-on-cell-execute --log-output > eval_client_students_vllm.log 2>&1
```